# FT-900 留出期原始月报采集

2023-01 至 2026-08 共 44 个月，1 份旧原件复用，最多 43 次新 GET。每年归档 NAS、远端哈希与限定恢复，零 SOXX 结果及模型。

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,os,shutil,tarfile,tempfile,urllib.request,subprocess,shlex
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/census_ft900_semiconductor_export_source_20261023'
plan=json.loads((card/'held_source_plan.json').read_text());manifest_path=card/plan['url_manifest'];manifest=json.loads(manifest_path.read_text())
assert hashlib.sha256(manifest_path.read_bytes()).hexdigest()==plan['url_manifest_sha256']
assert len(manifest['month_urls'])==44 and len(manifest['reuse'])==1
store=root/'data/snapshots/census_ft900_held_20261023';store.mkdir(parents=True,exist_ok=True)
ledger_path=card/'held_request_ledger.jsonl';archives_path=card/'held_year_archives.jsonl'
def append(path,row):
    with path.open('a') as f:f.write(json.dumps(row,sort_keys=True)+'\n');f.flush();os.fsync(f.fileno())
def read(path):return [json.loads(x) for x in path.read_text().splitlines()] if path.exists() else []
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
old=read(ledger_path);finished={x['month']:x for x in old if x['event'] in ('completed','reused')};started={x['month'] for x in old if x['event']=='started'}
assert len(finished)==len(set(finished)) and all(m in manifest['month_urls'] for m in finished)
if started-set(finished):raise RuntimeError('uncertain prior GET; do not retry: '+str(sorted(started-set(finished))))
new_requests=sum(x['event']=='started' for x in old);archives={x['year']:x for x in read(archives_path) if x['event']=='archive_verified'};settings=_settings(root)
for year in range(2023,2027):
    months=[f'{year:04d}-{m:02d}' for m in range(1,9 if year==2026 else 13)]
    if str(year) in archives:
        assert all(m in finished for m in months);continue
    failure=None
    for month in months:
        url=manifest['month_urls'][month];name=url.rsplit('/',1)[-1];dest=store/name
        if month in finished:assert dest.exists() and sha(dest)==finished[month]['sha256'];continue
        if month in manifest['reuse']:
            source=root/manifest['reuse'][month]['local_path'];expected=manifest['reuse'][month]['sha256'];assert sha(source)==expected
            if not dest.exists():shutil.copyfile(source,dest)
            assert sha(dest)==expected
            row={'event':'reused','month':month,'url':url,'source_path':str(source.relative_to(root)),'local_path':str(dest.relative_to(root)),'sha256':expected,'bytes':dest.stat().st_size};append(ledger_path,row);finished[month]=row;continue
        if new_requests>=plan['budget']['new_monthly_PDF_GETs_max']:raise RuntimeError('frozen GET budget exhausted')
        start={'event':'started','month':month,'url':url,'requested_at_utc':datetime.now(timezone.utc).isoformat()};append(ledger_path,start);new_requests+=1
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'SOXL-Qlib-source-audit/1.0','Accept':'application/pdf'})
            with urllib.request.urlopen(req,timeout=60) as response:
                status=response.status;headers={k:response.headers.get(k) for k in ('Date','Last-Modified','ETag','Content-Type','Content-Length','Cache-Control')};payload=response.read(4000001)
            if status!=200 or len(payload)>4000000 or not payload.startswith(b'%PDF-'):raise ValueError('invalid_HTTP_size_or_PDF_header')
            temporary=dest.with_suffix('.pdf.partial');temporary.write_bytes(payload);os.replace(temporary,dest)
            row={'event':'completed','month':month,'url':url,'requested_at_utc':start['requested_at_utc'],'received_at_utc':datetime.now(timezone.utc).isoformat(),'http_status':status,'headers':headers,'local_path':str(dest.relative_to(root)),'sha256':hashlib.sha256(payload).hexdigest(),'bytes':len(payload)};append(ledger_path,row);finished[month]=row
        except Exception as exc:
            failure={'event':'failed','month':month,'url':url,'error_type':type(exc).__name__,'error':str(exc),'observed_at_utc':datetime.now(timezone.utc).isoformat()};append(ledger_path,failure);break
    present=[m for m in months if m in finished]
    if present:
        with tempfile.TemporaryDirectory(prefix='ft900-held-') as tmp:
            archive=Path(tmp)/f'{year}.tar.gz'
            with tarfile.open(archive,'w:gz') as out:
                for month in present:
                    entry=finished[month];source=root/entry['local_path'];assert sha(source)==entry['sha256'];out.add(source,arcname=source.name)
            digest=sha(archive);size=archive.stat().st_size
            receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
            assert receipt['sha256']==digest and receipt['size_bytes']==size
            remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
            assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
            sample=present[0];entry=finished[sample];member=Path(entry['local_path']).name
            command=['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes','-o','ConnectTimeout=5',settings['ssh']['host'],'tar -xOzf '+repr(remote)+' '+repr(member)]
            restored=subprocess.run(command,capture_output=True,timeout=120);assert restored.returncode==0 and hashlib.sha256(restored.stdout).hexdigest()==entry['sha256'] and len(restored.stdout)==entry['bytes']
            row={'event':'archive_verified','year':str(year),'months':present,'archive_sha256':digest,'archive_bytes':size,'nas_artifact':receipt,'remote_hash_verified':True,'scoped_restore_month':sample,'scoped_restore_sha256':entry['sha256']};append(archives_path,row);archives[str(year)]=row
            print(year,len(present),'PDFs, NAS and scoped restore verified',flush=True)
    if failure:raise RuntimeError('source GET failed; do not retry: '+json.dumps(failure))
assert len(finished)==44 and new_requests==43 and len(archives)==4
result={'status':'all_44_held_month_PDF_bytes_archived_source_only','target_months':44,'reused_PDFs':1,'new_HTTP_GETs':new_requests,'NAS_archives':4,'scoped_restores':4,'all_remote_hashes_verified':True,'historical_first_public_bytes_qualified':False,'Exhibit7_row_coverage_qualified':False,'Qlib_factor_rows':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'SOXX_goal_achieved':False}
(card/'held_source_result.json').write_text(json.dumps(result,indent=2)+'\n');print(result['status'],flush=True)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


2023 12 PDFs, NAS and scoped restore verified


2024 12 PDFs, NAS and scoped restore verified


2025 12 PDFs, NAS and scoped restore verified


2026 8 PDFs, NAS and scoped restore verified


all_44_held_month_PDF_bytes_archived_source_only


## 结论边界

留出期 PDF 字节覆盖仅允许下一步原件行和报告月份审查。它不是独立策略前向日，也不证明历史首次公开版本或 Stage1 增量。

本次执行完成 44/44 个留出期月份 PDF 字节（1 份复用、43 次新 GET），4 个 NAS 包远端哈希通过、各限定恢复 1 份。历史首次公开字节、逐月 Exhibit 7 行与 Stage1 增量仍未获资格，0 模型/账户。